# Processed E-Commerce Dataset
### Combining Orders, Customers & Products with Pandas

**Input files:**
- `Day9_Orders.csv`
- `Day9_Customers.csv`
- `Day9_Products.csv`

This notebook loads the three related datasets, combines them using `merge()`,
demonstrates `concat()`, engineers new columns with `apply()`, performs
DateTime operations on the order date, and exports a single clean processed
dataset as a CSV file.

**Contents**
1. Setup & Data Loading
2. Initial Inspection
3. Merging DataFrames (`merge()`)
4. Combining DataFrames with `concat()`
5. DateTime Operations
6. Feature Engineering with `apply()`
7. Final Processed Dataset
8. Export to CSV
9. Observations


## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

orders = pd.read_csv('Day9_Orders.csv')
customers = pd.read_csv('Day9_Customers.csv')
products = pd.read_csv('Day9_Products.csv')

print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)
print("Products shape:", products.shape)


Orders shape: (120, 7)
Customers shape: (30, 5)
Products shape: (20, 5)


In [2]:
orders.head()

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


In [3]:
customers.head()

,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular
2,C003,Rohan Mehta,Mumbai,West,Premium
3,C004,Ananya Singh,Jammu,North,Regular
4,C005,Kabir Ali,Lucknow,North,New


In [4]:
products.head()

,Product_ID,Product_Name,Category,Unit_Price,Brand
0,P001,Wireless Headphones,Electronics,1499,SoundMax
1,P002,Mechanical Keyboard,Electronics,2499,KeyPro
2,P003,Wireless Mouse,Electronics,899,TechGear
3,P004,Smart Watch,Electronics,3299,FitTech
4,P005,Power Bank,Electronics,1199,VoltPlus


## 2. Initial Inspection

In [5]:
print("Orders info:")
orders.info()
print("\nMissing values in orders:\n", orders.isnull().sum())


Orders info:
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Order_ID        120 non-null    str  
 1   Order_Date      120 non-null    str  
 2   Customer_ID     120 non-null    str  
 3   Product_ID      120 non-null    str  
 4   Quantity        120 non-null    int64
 5   Payment_Method  120 non-null    str  
 6   Order_Status    120 non-null    str  
dtypes: int64(1), str(6)
memory usage: 6.7 KB

Missing values in orders:
 Order_ID          0
Order_Date        0
Customer_ID       0
Product_ID        0
Quantity          0
Payment_Method    0
Order_Status      0
dtype: int64


In [6]:
print("Customers info:")
customers.info()
print("\nMissing values in customers:\n", customers.isnull().sum())


Customers info:
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Customer_ID      30 non-null     str  
 1   Customer_Name    30 non-null     str  
 2   City             30 non-null     str  
 3   Region           30 non-null     str  
 4   Membership_Type  30 non-null     str  
dtypes: str(5)
memory usage: 1.3 KB

Missing values in customers:
 Customer_ID        0
Customer_Name      0
City               0
Region             0
Membership_Type    0
dtype: int64


In [7]:
print("Products info:")
products.info()
print("\nMissing values in products:\n", products.isnull().sum())


Products info:
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Product_ID    20 non-null     str  
 1   Product_Name  20 non-null     str  
 2   Category      20 non-null     str  
 3   Unit_Price    20 non-null     int64
 4   Brand         20 non-null     str  
dtypes: int64(1), str(4)
memory usage: 932.0 bytes

Missing values in products:
 Product_ID      0
Product_Name    0
Category        0
Unit_Price      0
Brand           0
dtype: int64


## 3. Merging DataFrames (`merge()`)

We combine the three tables using `Customer_ID` and `Product_ID` as keys,
joining `Orders` -> `Customers` -> `Products` with left merges so every
order is retained.

In [8]:
# Merge orders with customer details on Customer_ID
orders_customers = pd.merge(orders, customers, on='Customer_ID', how='left')
print("Shape after merging orders + customers:", orders_customers.shape)
orders_customers.head()


Shape after merging orders + customers: (120, 11)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Customer_Name,City,Region,Membership_Type
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,Harsh Vardhan,Noida,North,Premium
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,Ishita Gupta,Bengaluru,South,Premium
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,Karan Joshi,Chandigarh,North,Regular
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,Maryam Khan,Hyderabad,South,Regular
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,Reyansh Jain,Kolkata,East,New


In [9]:
# Merge the result with product details on Product_ID
merged_df = pd.merge(orders_customers, products, on='Product_ID', how='left')
print("Shape after merging with products:", merged_df.shape)
merged_df.head()


Shape after merging with products: (120, 15)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Customer_Name,City,Region,Membership_Type,Product_Name,Category,Unit_Price,Brand
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,Harsh Vardhan,Noida,North,Premium,Cricket Bat,Sports,2499,BatPro
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,Ishita Gupta,Bengaluru,South,Premium,Wireless Mouse,Electronics,899,TechGear
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,Karan Joshi,Chandigarh,North,Regular,Smart Watch,Electronics,3299,FitTech
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,Maryam Khan,Hyderabad,South,Regular,Machine Learning Basics,Books,999,AIPress
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,Reyansh Jain,Kolkata,East,New,Coffee Maker,Home & Kitchen,3499,HomeBrew


In [10]:
# Sanity check: no rows lost or duplicated during merges
assert len(merged_df) == len(orders), "Row count changed unexpectedly after merging!"
print("Merge verified: row count matches original Orders table:", len(merged_df))


Merge verified: row count matches original Orders table: 120


## 4. Combining DataFrames with `concat()`

`concat()` is typically used to stack DataFrames that share the same structure
(e.g. combining new batches of orders, or splitting/rejoining a dataset).
Here we demonstrate it by splitting the merged dataset into two halves and
stacking them back together with `pd.concat()`.

In [11]:
half = len(merged_df) // 2
first_half = merged_df.iloc[:half]
second_half = merged_df.iloc[half:]

# Recombine the two halves using concat()
concatenated_df = pd.concat([first_half, second_half], axis=0).reset_index(drop=True)

print("First half shape:", first_half.shape)
print("Second half shape:", second_half.shape)
print("Recombined shape:", concatenated_df.shape)
assert concatenated_df.shape == merged_df.shape
print("concat() demonstration verified: recombined dataset matches the merged dataset.")


First half shape: (60, 15)
Second half shape: (60, 15)
Recombined shape: (120, 15)
concat() demonstration verified: recombined dataset matches the merged dataset.


In [12]:
# Another concat() example: stacking Premium and Regular/New customer orders separately,
# then concatenating them back into one DataFrame
premium_orders = merged_df[merged_df['Membership_Type'] == 'Premium']
other_orders = merged_df[merged_df['Membership_Type'] != 'Premium']

recombined_by_membership = pd.concat([premium_orders, other_orders], axis=0).reset_index(drop=True)
print("Premium orders:", premium_orders.shape[0])
print("Other orders:", other_orders.shape[0])
print("Recombined total:", recombined_by_membership.shape[0])


Premium orders: 48
Other orders: 72
Recombined total: 120


## 5. DateTime Operations

Convert `Order_Date` to a proper datetime type and extract useful date
components: month, day, day name, and week of year.

In [13]:
merged_df['Order_Date'] = pd.to_datetime(merged_df['Order_Date'])

merged_df['Order_Month'] = merged_df['Order_Date'].dt.month
merged_df['Order_Month_Name'] = merged_df['Order_Date'].dt.month_name()
merged_df['Order_Day'] = merged_df['Order_Date'].dt.day
merged_df['Order_Day_Of_Week'] = merged_df['Order_Date'].dt.day_name()
merged_df['Order_Week'] = merged_df['Order_Date'].dt.isocalendar().week
merged_df['Is_Weekend'] = merged_df['Order_Day_Of_Week'].isin(['Saturday', 'Sunday'])

merged_df[['Order_ID', 'Order_Date', 'Order_Month_Name', 'Order_Day',
           'Order_Day_Of_Week', 'Order_Week', 'Is_Weekend']].head(10)


,Order_ID,Order_Date,Order_Month_Name,Order_Day,Order_Day_Of_Week,Order_Week,Is_Weekend
0,O0001,2026-02-19,February,19,Thursday,8,False
1,O0002,2026-01-25,January,25,Sunday,4,True
2,O0003,2026-02-26,February,26,Thursday,9,False
3,O0004,2026-03-04,March,4,Wednesday,10,False
4,O0005,2026-03-29,March,29,Sunday,13,True
5,O0006,2026-02-09,February,9,Monday,7,False
6,O0007,2026-02-10,February,10,Tuesday,7,False
7,O0008,2026-03-27,March,27,Friday,13,False
8,O0009,2026-03-13,March,13,Friday,11,False
9,O0010,2026-03-05,March,5,Thursday,10,False


In [14]:
# Orders per month
merged_df.groupby('Order_Month_Name')['Order_ID'].count().sort_values(ascending=False)


Order_Month_Name
January     44
February    41
March       35
Name: Order_ID, dtype: int64

In [15]:
# Orders per day of the week
merged_df['Order_Day_Of_Week'].value_counts()


Order_Day_Of_Week
Thursday     20
Friday       20
Sunday       18
Monday       17
Tuesday      17
Wednesday    14
Saturday     14
Name: count, dtype: int64

## 6. Feature Engineering with `apply()`

Create new, useful columns using `apply()`:
- `Total_Amount`: Quantity x Unit_Price
- `Order_Value_Segment`: categorize each order as Low / Medium / High value
- `Customer_Name_Initials`: initials derived from the customer's name
- `Is_Delivered`: boolean flag from Order_Status


In [16]:
# Total order value
merged_df['Total_Amount'] = merged_df.apply(lambda row: row['Quantity'] * row['Unit_Price'], axis=1)

merged_df[['Order_ID', 'Quantity', 'Unit_Price', 'Total_Amount']].head()


,Order_ID,Quantity,Unit_Price,Total_Amount
0,O0001,2,2499,4998
1,O0002,2,899,1798
2,O0003,1,3299,3299
3,O0004,3,999,2997
4,O0005,5,3499,17495


In [17]:
# Categorize orders into value segments using apply()
def segment_order(amount):
    if amount < 2000:
        return 'Low'
    elif amount < 6000:
        return 'Medium'
    else:
        return 'High'

merged_df['Order_Value_Segment'] = merged_df['Total_Amount'].apply(segment_order)
merged_df['Order_Value_Segment'].value_counts()


Order_Value_Segment
Medium    55
High      39
Low       26
Name: count, dtype: int64

In [18]:
# Derive customer initials using apply()
merged_df['Customer_Name_Initials'] = merged_df['Customer_Name'].apply(
    lambda name: ''.join([part[0].upper() for part in name.split()])
)

merged_df[['Customer_Name', 'Customer_Name_Initials']].drop_duplicates().head()


,Customer_Name,Customer_Name_Initials
0,Harsh Vardhan,HV
1,Ishita Gupta,IG
2,Karan Joshi,KJ
3,Maryam Khan,MK
4,Reyansh Jain,RJ


In [19]:
# Boolean flag for delivered orders using apply()
merged_df['Is_Delivered'] = merged_df['Order_Status'].apply(lambda status: status == 'Delivered')

merged_df['Is_Delivered'].value_counts()


Is_Delivered
True     79
False    41
Name: count, dtype: int64

## 7. Final Processed Dataset

Organize the columns into a clean, logically ordered DataFrame ready for
analysis or reporting.

In [20]:
final_columns = [
    'Order_ID', 'Order_Date', 'Order_Month_Name', 'Order_Day_Of_Week', 'Is_Weekend',
    'Customer_ID', 'Customer_Name', 'Customer_Name_Initials', 'City', 'Region', 'Membership_Type',
    'Product_ID', 'Product_Name', 'Category', 'Brand',
    'Quantity', 'Unit_Price', 'Total_Amount', 'Order_Value_Segment',
    'Payment_Method', 'Order_Status', 'Is_Delivered'
]

processed_df = merged_df[final_columns].sort_values(by='Order_Date').reset_index(drop=True)
processed_df.head(10)


,Order_ID,Order_Date,Order_Month_Name,Order_Day_Of_Week,Is_Weekend,Customer_ID,Customer_Name,Customer_Name_Initials,City,Region,Membership_Type,Product_ID,Product_Name,Category,Brand,Quantity,Unit_Price,Total_Amount,Order_Value_Segment,Payment_Method,Order_Status,Is_Delivered
0,O0051,2026-01-01,January,Thursday,False,C008,Sara Ahmed,SA,Hyderabad,South,Premium,P017,Yoga Mat,Sports,FitLife,4,899,3596,Medium,Net Banking,Delivered,True
1,O0091,2026-01-01,January,Thursday,False,C002,Zoya Khan,ZK,Delhi,North,Regular,P011,Air Fryer,Home & Kitchen,CookSmart,2,4999,9998,High,Credit Card,Delivered,True
2,O0022,2026-01-02,January,Friday,False,C023,Nikhil Sood,NS,Chandigarh,North,Premium,P014,Data Science Handbook,Books,DataPress,4,899,3596,Medium,Net Banking,Delivered,True
3,O0106,2026-01-02,January,Friday,False,C001,Aarav Sharma,AS,Srinagar,North,Premium,P009,Coffee Maker,Home & Kitchen,HomeBrew,1,3499,3499,Medium,Net Banking,Delivered,True
4,O0076,2026-01-03,January,Saturday,True,C019,Yusuf Dar,YD,Srinagar,North,New,P015,Machine Learning Basics,Books,AIPress,1,999,999,Low,Debit Card,Delivered,True
5,O0108,2026-01-04,January,Sunday,True,C006,Ishita Gupta,IG,Bengaluru,South,Premium,P020,Dumbbell Set,Sports,StrongFit,1,1999,1999,Low,Net Banking,Delivered,True
6,O0041,2026-01-04,January,Sunday,True,C008,Sara Ahmed,SA,Hyderabad,South,Premium,P012,Water Bottle,Home & Kitchen,HydroLife,5,699,3495,Medium,Cash on Delivery,Delivered,True
7,O0018,2026-01-05,January,Monday,False,C003,Rohan Mehta,RM,Mumbai,West,Premium,P011,Air Fryer,Home & Kitchen,CookSmart,4,4999,19996,High,Net Banking,Cancelled,False
8,O0104,2026-01-05,January,Monday,False,C011,Vivaan Kapoor,VK,Jaipur,North,New,P006,Hoodie,Clothing,UrbanWear,3,1599,4797,Medium,Debit Card,Shipped,False
9,O0110,2026-01-05,January,Monday,False,C007,Aditya Verma,AV,Pune,West,Regular,P005,Power Bank,Electronics,VoltPlus,1,1199,1199,Low,UPI,Delivered,True


In [21]:
print("Final processed dataset shape:", processed_df.shape)
processed_df.info()


Final processed dataset shape: (120, 22)
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Order_ID                120 non-null    str           
 1   Order_Date              120 non-null    datetime64[us]
 2   Order_Month_Name        120 non-null    str           
 3   Order_Day_Of_Week       120 non-null    str           
 4   Is_Weekend              120 non-null    bool          
 5   Customer_ID             120 non-null    str           
 6   Customer_Name           120 non-null    str           
 7   Customer_Name_Initials  120 non-null    str           
 8   City                    120 non-null    str           
 9   Region                  120 non-null    str           
 10  Membership_Type         120 non-null    str           
 11  Product_ID              120 non-null    str           
 12  Product_Name        

## 8. Export to CSV

In [22]:
processed_df.to_csv('Processed_Ecommerce_Dataset.csv', index=False)
print("Exported: Processed_Ecommerce_Dataset.csv")


Exported: Processed_Ecommerce_Dataset.csv


## 9. Observations

1. **Data integration:** Merging `Orders`, `Customers`, and `Products` on their
   respective ID columns produced a single unified dataset with no row loss,
   confirming clean, consistent keys across all three source files.
2. **Seasonality:** Grouping by `Order_Month_Name` and `Order_Day_Of_Week`
   reveals which months and weekdays see the most order activity — useful for
   planning inventory and staffing.
3. **Order value distribution:** The `Order_Value_Segment` column (Low/Medium/High,
   derived via `apply()`) shows how orders are distributed across spending
   tiers, which can inform targeted promotions for each segment.
4. **Membership impact:** Splitting orders by `Membership_Type` (Premium vs.
   Regular/New) and recombining with `concat()` makes it easy to compare
   ordering patterns between customer tiers.
5. **Delivery performance:** The `Is_Delivered` flag makes it straightforward
   to calculate delivery success rate and investigate cancelled/shipped orders
   separately.
6. **Clean export:** The final `Processed_Ecommerce_Dataset.csv` consolidates
   all relevant order, customer, and product attributes into one analysis-ready
   file, removing the need to repeatedly join the raw tables.
